## 0. Google Colab Setup

Mount Google Drive to access the project directory, and install dependencies. Run this cell first in every session to establish the working path.

### Mount Google Drive

Mount Google Drive to make the project directory available at `/content/drive/MyDrive/`. This must run before any path resolution or file access.

**Expected output:** A confirmation that Drive is already mounted or a prompt to authorize access.

In [12]:
import sys
import os
from google.colab import drive

# Mount Google Drive
drive.mount('/content/drive')

# Set working directory and append project root to sys.path
project_path = '/content/drive/MyDrive/multimodal-causal-ablation'
if os.path.exists(project_path):
    os.chdir(project_path)
    if project_path not in sys.path:
        sys.path.insert(0, project_path)
    print(f'Successfully set working directory and Python path to: {project_path}')
else:
    print(f'Folder not found: {project_path}')

# Install pinned dependencies
!pip install -r requirements.txt

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Successfully set working directory and Python path to: /content/drive/MyDrive/multimodal-causal-ablation


# Phase A: Dominant Modality Verification

Verify the dominant modality via aggregated DeepSHAP attribution, following the methodology locked in [ADR 0001](docs/adr/0001-causal-validation-methodology.md).

**Goal:** Formally confirm which modality contributes the highest aggregated DeepSHAP attribution score for both the base and fine-tuned models. This is the prerequisite gate before any neuron-level probing or ablation work in Phases B–D.

## 1. Environment & Imports

Set up the deterministic seed (`seed=0`, matching upstream checkpoint convention) and import all required libraries. The seed utility from `src/utils.py` pins `torch`, `numpy`, `random`, and `cudnn` for full reproducibility across ephemeral Colab runtimes.

**Expected output:** Confirmation of project path, checkpoints directory, and results directory.

In [13]:
import sys
import os
import pickle

import numpy as np
import pandas as pd

# Add project src to path for utility imports
sys.path.insert(0, os.path.join(project_path, 'src'))
from utils import set_deterministic_seed

# Pin all randomness sources (seed=0 matches upstream checkpoint convention)
set_deterministic_seed(seed=0)

# Define paths
checkpoints_dir = os.path.join(project_path, 'checkpoints')
results_dir = os.path.join(project_path, 'results')
os.makedirs(results_dir, exist_ok=True)

print(f'Project path:  {project_path}')
print(f'Checkpoints:   {checkpoints_dir}')
print(f'Results:       {results_dir}')

Project path:  /content/drive/MyDrive/multimodal-causal-ablation
Checkpoints:   /content/drive/MyDrive/multimodal-causal-ablation/checkpoints
Results:       /content/drive/MyDrive/multimodal-causal-ablation/results


## 2. Load Pre-computed DeepSHAP Attributions

Load the pre-computed DeepSHAP attribution pickles for both the base and fine-tuned models. Each pickle contains a dict with:
- `'SHAP_value'`: list of 6 numpy arrays (one per emotion class), each shaped `(n_samples, 1152)`
- `'test_feature'`: corresponding test input features

The 1152-dimensional feature vector is a concatenation of three modality representations:
- **Text** (ALBERT): dimensions 0–1023 (1024-d)
- **Video** (visual): dimensions 1024–1087 (64-d)
- **Audio** (acoustic): dimensions 1088–1151 (64-d)

**Expected output:** Structure summary showing keys, number of classes, and per-class array shapes for both models.

In [14]:
import os
import pickle

# Ensure project paths are set
project_path = '/content/drive/MyDrive/multimodal-causal-ablation'
if os.path.exists(project_path):
    os.chdir(project_path)

checkpoints_dir = os.path.join(os.getcwd(), 'checkpoints')

# Load pre-computed DeepSHAP attributions for both models
with open(os.path.join(checkpoints_dir, 'base_shap.pkl'), 'rb') as f:
    base_shap_data = pickle.load(f)

with open(os.path.join(checkpoints_dir, 'finetuned_shap.pkl'), 'rb') as f:
    finetuned_shap_data = pickle.load(f)

# Inspect structure
print('=== Base Model SHAP ===')
print(f'Keys: {list(base_shap_data.keys())}')
print(f'Number of classes: {len(base_shap_data["SHAP_value"])}')
for i, sv in enumerate(base_shap_data['SHAP_value']):
    print(f'  Class {i}: shape = {sv.shape}')

print()
print('=== Fine-tuned Model SHAP ===')
print(f'Keys: {list(finetuned_shap_data.keys())}')
print(f'Number of classes: {len(finetuned_shap_data["SHAP_value"])}')
for i, sv in enumerate(finetuned_shap_data['SHAP_value']):
    print(f'  Class {i}: shape = {sv.shape}')

=== Base Model SHAP ===
Keys: ['SHAP_value', 'test_feature']
Number of classes: 6
  Class 0: shape = (144, 1152)
  Class 1: shape = (144, 1152)
  Class 2: shape = (144, 1152)
  Class 3: shape = (144, 1152)
  Class 4: shape = (144, 1152)
  Class 5: shape = (144, 1152)

=== Fine-tuned Model SHAP ===
Keys: ['SHAP_value', 'test_feature']
Number of classes: 6
  Class 0: shape = (144, 1152)
  Class 1: shape = (144, 1152)
  Class 2: shape = (144, 1152)
  Class 3: shape = (144, 1152)
  Class 4: shape = (144, 1152)
  Class 5: shape = (144, 1152)


## 3. Compute Aggregated SHAP Attribution per Modality

Per ADR 0001 and the Section VI-D audit remediation, raw 1152-d SHAP values create a **dimensionality illusion** (Text has 1024 features vs. 64 for Audio/Video) and a **sign-cancellation hazard** (signed sums cancel out across dimensions).

To resolve this, I compute attributions using three complementary formulas, using `mean(|phi|)` as the primary decision criterion:
1. **`mean(|phi|)` (Primary Gatekeeper):** Average absolute attribution per feature (`np.mean(np.abs(sv))`). Dimension-invariant and free of sign-cancellation bias.
2. **`sum(|phi|)`:** Total absolute feature attribution (`np.sum(np.abs(sv))`).
3. **`|sum(phi)|`:** Net directional push (`np.abs(np.sum(sv))`).

`mean(|phi|)` dictates the dominant modality verdict for downstream Phase B/C/D targeting, and the verdict configuration is exported to `results/phase_a_dominant_modality_verdict.json`.

In [15]:
import sys
import os
import json
import pandas as pd

# Ensure project root is in sys.path for Colab runtimes
project_path = '/content/drive/MyDrive/multimodal-causal-ablation'
if os.path.exists(project_path) and project_path not in sys.path:
    sys.path.insert(0, project_path)

from src.metrics import compute_modality_attributions, resolve_dominant_modality_verdict

# Compute comprehensive attributions using modular src.metrics
verdict_payload = resolve_dominant_modality_verdict(
    base_shap=base_shap_data['SHAP_value'],
    ft_shap=finetuned_shap_data['SHAP_value'],
)

# Export dynamic configuration JSON for downstream phases
os.makedirs('results', exist_ok=True)
verdict_json_path = os.path.join('results', 'phase_a_dominant_modality_verdict.json')
with open(verdict_json_path, 'w') as f:
    json.dump(verdict_payload, f, indent=2)

print("\n=== Phase A Dominant Modality Verdict ===")
print(f"Primary Gatekeeper Metric: {verdict_payload['primary_criterion']}")
print(f"Base Model Winner:       {verdict_payload['base_model']['dominant_modality']}")
print(f"Fine-Tuned Model Winner:  {verdict_payload['finetuned_model']['dominant_modality']}")
print("\nBase Model Shares (%):")
for mod, share in verdict_payload['base_model']['mean_abs_phi_shares_percent'].items():
    print(f"  {mod:6s}: {share:6.2f}%")

print("\nFine-Tuned Model Shares (%):")
for mod, share in verdict_payload['finetuned_model']['mean_abs_phi_shares_percent'].items():
    print(f"  {mod:6s}: {share:6.2f}%")

print(f"\nSaved dynamic downstream config to: {verdict_json_path}")


=== Phase A Dominant Modality Verdict ===
Primary Gatekeeper Metric: mean_abs_phi
Base Model Winner:       Audio
Fine-Tuned Model Winner:  Audio

Base Model Shares (%):
  Text  :  14.56%
  Video :   9.52%
  Audio :  75.92%

Fine-Tuned Model Shares (%):
  Text  :  12.05%
  Video :   4.39%
  Audio :  83.56%

Saved dynamic downstream config to: results/phase_a_dominant_modality_verdict.json


### Apply Dominant Modality Decision Rule

Apply the ADR 0001 decision rule to the aggregated SHAP attributions to formally select the dominant modality. If the gap between the top two modalities is within 5%, the Audio branch (64-d) is chosen for its superior neuron-to-class ratio. Otherwise, the clear winner is selected.

**Expected output:** A printed verdict for both base and fine-tuned models, and two saved CSV files: `phase_a_dominant_modality_verdict.csv` and `phase_a_shap_attribution_by_class.csv`.

In [16]:
def compute_verdict(attr_df, model_name):
    """Apply ADR 0001 dominant modality decision rule.

    Returns a dict with the verdict and supporting evidence.
    """
    model_df = attr_df[attr_df['model'] == model_name]

    # Overall attribution: mean across classes
    summary = {}
    for mod_name in MODALITY_RANGES:
        summary[mod_name] = model_df[f'{mod_name}_attribution'].mean()

    total = sum(summary.values())
    pct = {k: v / total * 100 for k, v in summary.items()}

    # Rank by attribution
    ranked = sorted(pct.items(), key=lambda x: x[1], reverse=True)

    sep = '=' * 55
    print(sep)
    print(f'  {model_name.upper()} MODEL — Aggregated SHAP Attribution')
    print(sep)
    for mod, p in ranked:
        print(f'  {mod:8s}: {p:6.2f}%  (raw mean: {summary[mod]:.6f})')

    top_mod, top_pct = ranked[0]
    second_mod, second_pct = ranked[1]
    gap = top_pct - second_pct

    # ADR 0001 decision rule
    if gap <= 5.0:
        dominant = 'Audio'
        reason = (
            f'Top two modalities ({top_mod}: {top_pct:.2f}%, '
            f'{second_mod}: {second_pct:.2f}%) are within 5% parity '
            f'(gap = {gap:.2f}%). Per ADR 0001, targeting Audio (64-d) '
            f'for superior neuron-to-class ratio.'
        )
    else:
        dominant = top_mod
        reason = (
            f'{top_mod} leads with {top_pct:.2f}% vs '
            f'{second_mod} at {second_pct:.2f}% '
            f'(gap = {gap:.2f}% > 5% threshold).'
        )

    print()
    print(f'  VERDICT: Dominant Modality = {dominant}')
    print(f'  Reason:  {reason}')

    return {
        'model': model_name,
        'dominant_modality': dominant,
        'Text_pct': round(pct['Text'], 4),
        'Video_pct': round(pct['Video'], 4),
        'Audio_pct': round(pct['Audio'], 4),
        'top_modality': top_mod,
        'second_modality': second_mod,
        'gap_pct': round(gap, 4),
        'parity_rule_applied': gap <= 5.0,
        'reason': reason,
    }


# Apply verdict to both models
base_verdict = compute_verdict(attribution_df, 'base')
print()
finetuned_verdict = compute_verdict(attribution_df, 'finetuned')

# --- Save results ---
verdict_df = pd.DataFrame([base_verdict, finetuned_verdict])
verdict_path = os.path.join(
    results_dir, 'phase_a_dominant_modality_verdict.csv'
)
verdict_df.to_csv(verdict_path, index=False)

detail_path = os.path.join(
    results_dir, 'phase_a_shap_attribution_by_class.csv'
)
attribution_df.to_csv(detail_path, index=False)

print()
print(f'Results saved:')
print(f'  Verdict:  {verdict_path}')
print(f'  Details:  {detail_path}')

# --- Summary ---
sep = '=' * 55
print()
print(sep)
print('  PHASE A SUMMARY')
print(sep)
bm = base_verdict['dominant_modality']
fm = finetuned_verdict['dominant_modality']
print(f'  Base model dominant modality:       {bm}')
print(f'  Fine-tuned model dominant modality:  {fm}')

if base_verdict['dominant_modality'] == 'Audio':
    print()
    print('  ✓ Audio confirmed as dominant modality.')
    print('    Proceed to Phase B: Probe Signal Validation '
          'on Audio FFN activations.')
else:
    dm = base_verdict['dominant_modality']
    print()
    print(f'  ✗ Audio NOT confirmed. Dominant modality = {dm}.')
    print('    Review methodology — experiment targets '
          'the dominant modality.')

  BASE MODEL — Aggregated SHAP Attribution
  Audio   :  50.48%  (raw mean: 0.040532)
  Text    :  43.66%  (raw mean: 0.035061)
  Video   :   5.86%  (raw mean: 0.004707)

  VERDICT: Dominant Modality = Audio
  Reason:  Audio leads with 50.48% vs Text at 43.66% (gap = 6.81% > 5% threshold).

  FINETUNED MODEL — Aggregated SHAP Attribution
  Audio   :  58.51%  (raw mean: 0.048210)
  Text    :  36.15%  (raw mean: 0.029787)
  Video   :   5.34%  (raw mean: 0.004396)

  VERDICT: Dominant Modality = Audio
  Reason:  Audio leads with 58.51% vs Text at 36.15% (gap = 22.36% > 5% threshold).

Results saved:
  Verdict:  /content/drive/MyDrive/multimodal-causal-ablation/results/phase_a_dominant_modality_verdict.csv
  Details:  /content/drive/MyDrive/multimodal-causal-ablation/results/phase_a_shap_attribution_by_class.csv

  PHASE A SUMMARY
  Base model dominant modality:       Audio
  Fine-tuned model dominant modality:  Audio

  ✓ Audio confirmed as dominant modality.
    Proceed to Phase B: Probe 

## 3.5 Full-Dataset Activation Extraction (P1 Fix)

The SHAP pickles only cached 144 test samples. Per the experiment protocol (Day 1-2), activation statistics and ablation evaluations should use the full RML dataset (train+val+test combined = 723 samples) since we are not training anything, just observing neuron behavior and computing selectivity statistics.

This cell loads the complete RML dataset, runs `SHAP_feature()` through both models, and caches the 1152-d representations. All downstream phases (B, C, D) will use these larger tensors.

**Expected output:** Confirmation of 723-sample feature tensors for both models, saved to `checkpoints/activations/`.

In [17]:
import os
import sys
import pickle
import numpy as np
import torch
from torch.utils.data import DataLoader

# Ensure src and model code are importable
sys.path.insert(0, os.path.join(project_path, 'Model/Dig-Data_Model-Main'))
sys.path.insert(0, os.path.join(project_path, 'src'))

from utils import set_deterministic_seed
set_deterministic_seed(seed=0)

from src.datasets import IEMOCAP, collate_fn
from src.models.e2e import MME2E
from transformers import AlbertTokenizer

device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

# ── Paths ──
data_dir = os.path.join(project_path, 'Model/Dig-Data_Model-Main/data')
main_folder = os.path.join(data_dir, 'RML_RAW_PROCESSED_Face')
split_dir = os.path.join(
    data_dir, 'data_split', 'all_single_label_six_category', 'with_valid'
)
activations_dir = os.path.join(project_path, 'checkpoints', 'activations')
os.makedirs(activations_dir, exist_ok=True)

# ── Load ALL split utterance IDs ──
train_ids = open(os.path.join(
    split_dir, 'Final_train_split_six_categories_RML.txt'
)).read().splitlines()
valid_ids = open(os.path.join(
    split_dir, 'Final_valid_split_six_categories_RML.txt'
)).read().splitlines()
test_ids = open(os.path.join(
    split_dir, 'Final_test_split_six_categories_RML.txt'
)).read().splitlines()
full_uttr_ids = train_ids + valid_ids + test_ids

print(f"Train: {len(train_ids)}, Val: {len(valid_ids)}, "
      f"Test: {len(test_ids)}, Total: {len(full_uttr_ids)}")

# ── Load metadata ──
with open(os.path.join(main_folder, 'meta.pkl'), 'rb') as f:
    meta = pickle.load(f)

emoDict = {'ang': 0, 'dis': 1, 'fea': 2, 'hap': 3, 'sad': 4, 'sur': 5}
# Handle missing transcripts (nan) gracefully
texts = [meta[uid]['text'] if isinstance(meta[uid]['text'], str) else "" for uid in full_uttr_ids]
labels_onehot = [np.eye(6)[emoDict[meta[uid]['label']]] for uid in full_uttr_ids]

# ── Build Dataset and DataLoader ──
full_dataset = IEMOCAP(
    main_folder=main_folder,
    utterance_ids=full_uttr_ids,
    texts=texts,
    labels=labels_onehot,
    label_annotations=list(emoDict.keys()),
    img_interval=500
)

full_loader = DataLoader(
    full_dataset,
    batch_size=8,
    shuffle=False,
    num_workers=2,
    pin_memory=True,
    collate_fn=collate_fn
)

print(f"Dataset size: {len(full_dataset)}")
print(f"DataLoader batches: {len(full_loader)}")

Device: cuda:0
Train: 518, Val: 58, Test: 144, Total: 720
Dataset size: 720
DataLoader batches: 90


/content/drive/MyDrive/multimodal-causal-ablation/Model/Dig-Data_Model-Main/src/datasets.py:268: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at ../torch/csrc/utils/tensor_new.cpp:275.)
  self.labels_no_onehot = torch.tensor(labels).argmax(-1)


### 3.5.1 Forward Pass: Extract Full-Dataset SHAP Features

Run `SHAP_feature()` through both the base and fine-tuned models for every sample in the full RML dataset. This produces two `(723, 1152)` tensors, the exact pre-classification representations that feed into `t_out`, `v_out`, `a_out`.

**Expected output:** Per-model progress bars and final tensor shapes. Estimated runtime: ~10-30 minutes per model depending on GPU.

In [18]:
import os
import numpy as np
import torch
from transformers import AlbertTokenizer

from src.models.e2e import MME2E

# ── 1. Top-Level Resume Gate Check ──
activations_dir = os.path.join(project_path, 'checkpoints', 'activations')
os.makedirs(activations_dir, exist_ok=True)

base_cache_path = os.path.join(activations_dir, 'base_full_1152.npy')
ft_cache_path = os.path.join(activations_dir, 'ft_full_1152.npy')
labels_cache_path = os.path.join(activations_dir, 'labels_full.npy')

# Set to True only if you explicitly want to re-run forward passes from scratch
force_reextract: bool = False

if (
    os.path.exists(base_cache_path)
    and os.path.exists(ft_cache_path)
    and os.path.exists(labels_cache_path)
    and not force_reextract
):
    print(
        '✓ Complete full-dataset activations found in checkpoints/activations/'
    )
    print(
        '  Loading cached tensors directly from disk (bypassing forward'
        ' pass)...'
    )
    base_full_1152 = np.load(base_cache_path)
    ft_full_1152 = np.load(ft_cache_path)
    labels_full = np.load(labels_cache_path)
    print(f'  base_full_1152 shape: {base_full_1152.shape}')
    print(f'  ft_full_1152 shape:   {ft_full_1152.shape}')
    print(f'  labels_full shape:    {labels_full.shape}')
else:
    print('Executing fresh feature extraction forward pass...')

    tokenizer = AlbertTokenizer.from_pretrained('albert-large-v2')
    model_args = {
        'num_emotions': 6,
        'modalities': 'tav',
        'feature_dim': 256,
        'trans_nlayers': 4,
        'trans_nheads': 4,
        'trans_dim': 64,
        'text_model_size': 'large',
        'text_max_len': 100,
    }

    def extract_full_features(
        model_path: str,
        model_name: str,
    ) -> tuple[np.ndarray, np.ndarray]:
        """Run SHAP_feature() over full dataset directly in memory."""
        print(f"\n{'='*55}")
        print(f'  Extracting features: {model_name}')
        print(f"{'='*55}")

        model = MME2E(args=model_args, device=device).to(device)
        model.load_state_dict(
            torch.load(model_path, map_location=device), strict=False
        )
        model.eval()

        all_features = []
        all_labels = []
        processed = 0

        for batch_idx, batch in enumerate(full_loader):
            uttr_ids_batch, imgs, img_lens, specs, spec_lens, text_batch, Y = (
                batch
            )

            with torch.no_grad():
                text_inputs = tokenizer(
                    list(text_batch),
                    return_tensors='pt',
                    max_length=model_args['text_max_len'],
                    padding='max_length',
                    truncation=True,
                ).to(device)

                specs = specs.to(device)
                features = model.SHAP_feature(
                    imgs, img_lens, specs, spec_lens, text_inputs
                )

                features_np = features.cpu().numpy()
                labels_np = (
                    Y.cpu().numpy() if hasattr(Y, 'cpu') else np.array(Y)
                )

            all_features.append(features_np)
            all_labels.append(labels_np)

            processed += len(uttr_ids_batch)
            if (batch_idx + 1) % 10 == 0 or (batch_idx + 1) == len(full_loader):
                print(
                    f'  Batch {batch_idx+1}/{len(full_loader)} '
                    f'({processed}/{len(full_dataset)} samples)'
                )

        all_features = np.concatenate(all_features, axis=0)
        all_labels = np.concatenate(all_labels, axis=0)
        int_labels = np.argmax(all_labels, axis=1)

        print(f'  Features shape: {all_features.shape}')
        print(f'  Labels shape:   {int_labels.shape}')

        del model
        torch.cuda.empty_cache()

        return all_features, int_labels

    base_full_1152, labels_full = extract_full_features(
        os.path.join(project_path, 'checkpoints', 'base_model.pt'),
        'Base Model'
    )

    ft_full_1152, labels_full_ft = extract_full_features(
        os.path.join(project_path, 'checkpoints', 'finetuned_model.pt'),
        'Fine-tuned Model'
    )

    assert np.array_equal(labels_full, labels_full_ft), (
        'Label ordering mismatch between models!'
    )
    print(
        '\n✓ Label ordering verified: both models processed identical samples.'
    )

    # Save combined activation checkpoints to disk for future sessions
    np.save(base_cache_path, base_full_1152)
    np.save(ft_cache_path, ft_full_1152)
    np.save(labels_cache_path, labels_full)

: 

: 

### 3.5.2 Cache Full-Dataset Activations

Save the extracted features and labels to `checkpoints/activations/` for resume-gate loading in future sessions. Then overwrite the in-memory variables used by Phases B, C, and D so downstream cells automatically use the full dataset.

**Expected output:** Saved file paths and updated variable shapes.

In [19]:
# ── Save to disk ──
np.save(os.path.join(activations_dir, 'base_full_1152.npy'), base_full_1152)
np.save(os.path.join(activations_dir, 'ft_full_1152.npy'), ft_full_1152)
np.save(os.path.join(activations_dir, 'labels_full.npy'), labels_full)

print("Saved to checkpoints/activations/:")
print(f"  base_full_1152.npy  -> {base_full_1152.shape}")
print(f"  ft_full_1152.npy    -> {ft_full_1152.shape}")
print(f"  labels_full.npy     -> {labels_full.shape}")

# ── Overwrite in-memory variables for downstream Phases ──
# These variable names are what Phase B/C/D cells expect
base_acts_tier1 = base_full_1152[:, 1088:1152]
ft_acts_tier1   = ft_full_1152[:, 1088:1152]
labels_tier1    = labels_full
base_1152       = base_full_1152
ft_1152         = ft_full_1152

print(f"\nIn-memory variables updated for downstream phases:")
print(f"  base_acts_tier1: {base_acts_tier1.shape}")
print(f"  ft_acts_tier1:   {ft_acts_tier1.shape}")
print(f"  labels_tier1:    {labels_tier1.shape}")
print(f"  base_1152:       {base_1152.shape}")
print(f"  ft_1152:         {ft_1152.shape}")

# ── Class distribution check ──
print(f"\nPer-class sample counts (full dataset):")
for c, name in enumerate(EMOTION_CLASSES):
    count = np.sum(labels_full == c)
    print(f"  {name:10s}: {count} samples ({count/len(labels_full)*100:.1f}%)")

Saved to checkpoints/activations/:
  base_full_1152.npy  -> (720, 1152)
  ft_full_1152.npy    -> (720, 1152)
  labels_full.npy     -> (720,)

In-memory variables updated for downstream phases:
  base_acts_tier1: (720, 64)
  ft_acts_tier1:   (720, 64)
  labels_tier1:    (720,)
  base_1152:       (720, 1152)
  ft_1152:         (720, 1152)

Per-class sample counts (full dataset):
  anger     : 120 samples (16.7%)
  disgust   : 120 samples (16.7%)
  fear      : 120 samples (16.7%)
  happiness : 120 samples (16.7%)
  sadness   : 120 samples (16.7%)
  surprise  : 120 samples (16.7%)


# Phase B: Probe Signal Validation

Validate the target modality's signal via L1-logistic regression, following the methodology locked in [ADR
0001](docs/adr/0001-causal-validation-methodology.md).

**Goal:** Ensure the chosen modality branch (Text, 1024-d) retains sufficient class-discriminative information
before we perform causal ablations. If the signal is too weak, we trigger a layer fallback.

## 4. Resume Gate: Load Cached Activations
Loads cached 1024-d Text branch activations to bypass expensive PyTorch forward passes.

**Expected output:** Confirmation of loaded tensor shapes.

In [22]:
import sys
import os
import json
import numpy as np

# Ensure project root is in sys.path
project_path = '/content/drive/MyDrive/multimodal-causal-ablation'
if os.path.exists(project_path):
    os.chdir(project_path)
    if project_path not in sys.path:
        sys.path.insert(0, project_path)

# Load Phase A dominant modality verdict JSON
verdict_path = os.path.join('results', 'phase_a_dominant_modality_verdict.json')
with open(verdict_path, 'r') as f:
    verdict_data = json.load(f)

downstream_cfg = verdict_data['downstream_config']
target_modality = downstream_cfg['target_modality']
slice_start = downstream_cfg['slice_start']
slice_end = downstream_cfg['slice_end']
feature_dim = downstream_cfg['feature_dim']

print(f"Loaded Phase A Verdict: Target Modality = {target_modality} (Slice [{slice_start}:{slice_end}], {feature_dim}-d)")

# Load full 1152-d representations (720 samples) from checkpoints/activations/
checkpoints_dir = os.path.join(os.getcwd(), 'checkpoints')
activations_dir = os.path.join(checkpoints_dir, 'activations')

base_full_1152 = np.load(os.path.join(activations_dir, 'base_full_1152.npy'))
ft_full_1152   = np.load(os.path.join(activations_dir, 'ft_full_1152.npy'))

# Load labels (720 samples)
if os.path.exists(os.path.join(activations_dir, 'labels_full.npy')):
    labels_tier1 = np.load(os.path.join(activations_dir, 'labels_full.npy'))
else:
    labels_tier1 = np.load(os.path.join(activations_dir, 'labels_720.npy'))

# Dynamically slice dominant modality representations (Tier 1)
base_acts_tier1 = base_full_1152[:, slice_start:slice_end]
ft_acts_tier1   = ft_full_1152[:, slice_start:slice_end]

print(f"Base Tier 1 activations shape: {base_acts_tier1.shape}")
print(f"FT Tier 1 activations shape:   {ft_acts_tier1.shape}")
print(f"Labels shape:                 {labels_tier1.shape}")

Loaded Phase A Verdict: Target Modality = Audio (Slice [1088:1152], 64-d)
Base Tier 1 activations shape: (720, 64)
FT Tier 1 activations shape:   (720, 64)
Labels shape:                 (720,)


## 5. L1-Logistic Probe Validation (Tier 1)

Below, we extract the correct 1024-d text features and the corresponding 144 RML true labels to perform the Tier 1 validation.

**Expected output:** Per-class AUC scores, Mean AUC, and a PASS/WARNING fallback decision for both the base and fine-tuned models.

In [23]:
import os
import numpy as np

checkpoints_dir = os.path.join(project_path, 'checkpoints')
activations_dir = os.path.join(checkpoints_dir, 'activations')

# Use full-dataset activations (720 samples) from 3.5.2 cache
if 'base_acts_tier1' not in globals() or len(base_acts_tier1) != 720:
    print("Loading 720-sample full dataset activations from disk cache...")
    base_acts_tier1 = np.load(os.path.join(activations_dir, 'base_full_1152.npy'))[:, 1088:1152]
    ft_acts_tier1 = np.load(os.path.join(activations_dir, 'ft_full_1152.npy'))[:, 1088:1152]

# Save pristine copies of full-dataset Tier 1 representations
np.save(os.path.join(activations_dir, 'base_acts_tier1.npy'), base_acts_tier1)
np.save(os.path.join(activations_dir, 'ft_acts_tier1.npy'), ft_acts_tier1)

print(f"Base Tier 1 shape: {base_acts_tier1.shape}")
print(f"FT Tier 1 shape:   {ft_acts_tier1.shape}")

Base Tier 1 shape: (720, 64)
FT Tier 1 shape:   (720, 64)


### 5.1 Aligning Target Labels
The previously cached labels were tied to the corrupted 288-sample dataset. Here, we cleanly extract the true 144 target labels for the RML dataset by matching the test split utterance IDs directly against the dataset's `meta.pkl`.

In [24]:
import os
import numpy as np

activations_dir = os.path.join(project_path, 'checkpoints', 'activations')

# Use full-dataset labels (720 samples) from 3.5.2 cache
if 'labels_tier1' not in globals() or len(labels_tier1) != 720:
    print("Loading 720-sample full dataset labels from disk cache...")
    labels_tier1 = np.load(os.path.join(activations_dir, 'labels_full.npy'))

print(f"Labels shape: {labels_tier1.shape}")

Labels shape: (720,)


### 5.2 Out-of-Fold Evaluation
We define and run our out-of-fold `StratifiedKFold` logistic regression pipeline on the clean Tier 1 representations. If the Mean AUC exceeds 0.65 and at least 4 out of 6 classes exceed 0.55, the signal validation formally passes. Validation results are then saved to a CSV artifact.

**Expected output:** Per-class AUC scores, Mean AUC, PASS/WARNING fallback decisions, and CSV artifact paths.

In [25]:
import os
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from utils import set_deterministic_seed

EMOTION_CLASSES = [
    'anger',
    'disgust',
    'fear',
    'happiness',
    'sadness',
    'surprise',
]


def validate_probe_signal(acts, labels, model_name):
  set_deterministic_seed(seed=0)
  print(f'=== {model_name.upper()} MODEL PROBE VALIDATION (Tier 1) ===')
  n_classes = len(np.unique(labels))
  cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)

  aucs = []
  results = []
  fitted_probes = {}

  for c in range(n_classes):
    y_binary = (labels == c).astype(int)

    # Scikit-learn Pipeline prevents pre-processing data leakage across CV folds (Issue B.3)
    pipe = Pipeline([
        ('scaler', StandardScaler()),
        (
            'probe',
            LogisticRegression(
                penalty='l1',
                solver='liblinear',
                class_weight='balanced',
                random_state=0,
                max_iter=1000,
            ),
        ),
    ])

    # Out-of-fold probability estimates for honest AUC evaluation
    probs = cross_val_predict(
        pipe, acts, y_binary, cv=cv, method='predict_proba'
    )[:, 1]
    auc = roc_auc_score(y_binary, probs)
    aucs.append(auc)
    results.append(
        {'model': model_name, 'class': EMOTION_CLASSES[c], 'auc': round(auc, 4)}
    )

    # Fit persistent probe on full data to retain L1 coefficients (Issue B.4)
    pipe.fit(acts, y_binary)
    fitted_probes[c] = pipe.named_steps['probe']

    print(f'  Class {EMOTION_CLASSES[c]:<10} AUC: {auc:.4f}')

  mean_auc = np.mean(aucs)
  poor_classes = sum(1 for a in aucs if a < 0.55)
  print('-' * 45)
  print(f'  Mean AUC: {mean_auc:.4f} (Threshold: >= 0.65)')
  print(f'  Classes < 0.55 AUC: {poor_classes} (Threshold: <= 2)')

  # Save CSV artifact
  df = pd.DataFrame(results)
  out_path = os.path.join(results_dir, f'phase_b_{model_name}_tier1_aucs.csv')
  df.to_csv(out_path, index=False)
  print(f'  [SAVED] {out_path}')

  return fitted_probes, (mean_auc >= 0.65 and poor_classes <= 2)


print('Validating True Tier 1 Signal...\n')
base_probes, base_valid = validate_probe_signal(
    base_acts_tier1, labels_tier1, 'base'
)
print()
ft_probes, ft_valid = validate_probe_signal(
    ft_acts_tier1, labels_tier1, 'finetuned'
)

Validating True Tier 1 Signal...

=== BASE MODEL PROBE VALIDATION (Tier 1) ===
  Class anger      AUC: 0.9394
  Class disgust    AUC: 0.8761
  Class fear       AUC: 0.7508
  Class happiness  AUC: 0.8002
  Class sadness    AUC: 0.9091
  Class surprise   AUC: 0.8262
---------------------------------------------
  Mean AUC: 0.8503 (Threshold: >= 0.65)
  Classes < 0.55 AUC: 0 (Threshold: <= 2)
  [SAVED] /content/drive/MyDrive/multimodal-causal-ablation/results/phase_b_base_tier1_aucs.csv

=== FINETUNED MODEL PROBE VALIDATION (Tier 1) ===
  Class anger      AUC: 0.9287
  Class disgust    AUC: 0.8200
  Class fear       AUC: 0.7082
  Class happiness  AUC: 0.8154
  Class sadness    AUC: 0.9179
  Class surprise   AUC: 0.8429
---------------------------------------------
  Mean AUC: 0.8389 (Threshold: >= 0.65)
  Classes < 0.55 AUC: 0 (Threshold: <= 2)
  [SAVED] /content/drive/MyDrive/multimodal-causal-ablation/results/phase_b_finetuned_tier1_aucs.csv


# Phase C: Causal Ablation

Compute class-selectivity ratios and perform mean-ablation sweeps, following the methodology locked in [ADR 0001](docs/adr/0001-causal-validation-methodology.md).

**Goal:** Identify the most causally active neurons for each emotion class in the 1024-d Text Tier 1 representation, and evaluate the classification accuracy drop when these specific neurons are ablated. A feature set is causally class-selective if the target class accuracy drop is $\ge 2.5\times$ the mean absolute non-target class drop.

## 6. Compute Class-Selectivity Ratios

To find the top $k$ neurons to ablate, we compute the selectivity ratio $\frac{\mu(n|c)}{\mu(n|\neg c)}$ for every neuron $n$ in the 1024-d representation across all emotion classes.

**Expected output:** A list of the top 5 highly selective neuron indices for each class in both the base and fine-tuned models.

In [26]:
import numpy as np
from utils import set_deterministic_seed

EMOTION_CLASSES = [
    'anger',
    'disgust',
    'fear',
    'happiness',
    'sadness',
    'surprise',
]
K_VALUES = [1, 3, 5, 10]

set_deterministic_seed(seed=0)
print('=== Phase C: Ranking Neurons by Absolute L1 Probe Weights ===')


def rank_top_neurons_by_l1_weights(fitted_probes):
  """Ranks neurons by absolute L1 logistic regression probe weight magnitude (|coef_|).

  This directly identifies the features the linear classifier relies on for
  detecting each emotion (resolving Issue B.1 & protocol Day 3/11-12
  requirements).
  """
  top_neurons = {}
  top_weights = {}

  for c, probe in fitted_probes.items():
    # probe.coef_ has shape (1, n_features) for binary classification
    coefs = np.abs(probe.coef_[0])

    # Rank indices in descending order of absolute weight magnitude
    ranked_indices = np.argsort(coefs)[::-1]
    top_neurons[c] = ranked_indices
    top_weights[c] = coefs[ranked_indices]

  return top_neurons, top_weights


# Compute rankings using fitted L1 probes from Phase B
base_top_neurons, base_top_weights = rank_top_neurons_by_l1_weights(
    base_probes
)
ft_top_neurons, ft_top_weights = rank_top_neurons_by_l1_weights(ft_probes)

print('\nBase Model — Top 5 Neurons per Class (L1 Probe Weight Ranking):')
for c, idxs in base_top_neurons.items():
  weights_str = ', '.join([f'{w:.4f}' for w in base_top_weights[c][:5]])
  print(
      f'  {EMOTION_CLASSES[c]:<10}: Indices={idxs[:5]} | Weights=[{weights_str}]'
  )

print('\nFine-Tuned Model — Top 5 Neurons per Class (L1 Probe Weight Ranking):')
for c, idxs in ft_top_neurons.items():
  weights_str = ', '.join([f'{w:.4f}' for w in ft_top_weights[c][:5]])
  print(
      f'  {EMOTION_CLASSES[c]:<10}: Indices={idxs[:5]} | Weights=[{weights_str}]'
  )

weight_rows = []
for c in range(6):
  class_name = EMOTION_CLASSES[c]
  for rank in range(5):
    b_idx = base_top_neurons[c][rank]
    b_w = base_top_weights[c][rank]
    ft_idx = ft_top_neurons[c][rank]
    ft_w = ft_top_weights[c][rank]

    weight_rows.append({
        'class': class_name,
        'rank': rank + 1,
        'base_neuron_idx': b_idx,
        'base_l1_weight': round(b_w, 4),
        'ft_neuron_idx': ft_idx,
        'ft_l1_weight': round(ft_w, 4),
    })

df_weights = pd.DataFrame(weight_rows)
out_weights_path = os.path.join(results_dir, 'phase_b_top5_probe_weights.csv')
df_weights.to_csv(out_weights_path, index=False)
print(f'\n[SAVED] Day 5 Probe Weight Table: {out_weights_path}')

=== Phase C: Ranking Neurons by Absolute L1 Probe Weights ===

Base Model — Top 5 Neurons per Class (L1 Probe Weight Ranking):
  anger     : Indices=[43  6 39 62 27] | Weights=[3.2385, 2.7412, 2.0972, 1.4652, 1.3043]
  disgust   : Indices=[61 44 30 50 53] | Weights=[3.8450, 2.5018, 2.0088, 1.9947, 1.9158]
  fear      : Indices=[44 43 41 56 20] | Weights=[3.2708, 3.0336, 1.8148, 1.6149, 1.5966]
  happiness : Indices=[18 49  6 27 44] | Weights=[2.3329, 2.3304, 2.0240, 1.9780, 1.9544]
  sadness   : Indices=[31 54 52  7 26] | Weights=[4.5920, 2.2261, 2.1592, 2.1025, 1.7403]
  surprise  : Indices=[62 35 30 45  8] | Weights=[2.7551, 2.5670, 2.3466, 1.6748, 1.6718]

Fine-Tuned Model — Top 5 Neurons per Class (L1 Probe Weight Ranking):
  anger     : Indices=[55 37 45  4 36] | Weights=[2.3939, 2.0902, 2.0837, 1.9794, 1.6122]
  disgust   : Indices=[32 26  1 63 31] | Weights=[4.3908, 3.9378, 3.7077, 3.6441, 3.0839]
  fear      : Indices=[43 27 20 13 31] | Weights=[2.6324, 2.1896, 2.0619, 2.0438, 

## 7. Mean-Ablation Proxy Inference & Model Loading

Because causal ablation must be measured via accuracy drops, we must evaluate the actual PyTorch `MME2E` models. However, instead of reloading the raw dataset (images, audio, text) and running the heavy encoders, we can leverage the exact 1152-d `test_feature` representations we already cached in Phase A.

The `MME2E` architecture allows us to cleanly split this 1152-d vector back into `Text (1024-d)`, `Video (64-d)`, and `Audio (64-d)`, and pass them directly into the pre-trained classification heads (`t_out`, `v_out`, `a_out`, and `weighted_fusion`). This "proxy inference" mathematically perfectly simulates the forward pass while allowing us to seamlessly clamp the Text neurons.

**Expected output:** Loading of the base and fine-tuned `MME2E` PyTorch checkpoints, and definition of the `ablate_and_evaluate` proxy inference function.

In [27]:
from utils import set_deterministic_seed
set_deterministic_seed(seed=0)
import sys
import os
import torch
import torch.nn as nn
import numpy as np

# Ensure MME2E is in path
sys.path.insert(0, os.path.join(project_path, 'Model/Dig-Data_Model-Main'))
from src.models.e2e import MME2E

# Setup mocked args for model instantiation (matches Phase A setup)
args = {
    'num_emotions': 6,
    'modalities': 'tav',
    'feature_dim': 256,
    'trans_nlayers': 4,
    'trans_nheads': 4,
    'trans_dim': 64,
    'text_model_size': 'large',
}
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')

print("Loading PyTorch MME2E models for ablation...")
base_model = MME2E(args=args, device=device).to(device)
base_model.load_state_dict(torch.load(os.path.join(checkpoints_dir, 'base_model.pt'), map_location=device), strict=False)
base_model.eval()

ft_model = MME2E(args=args, device=device).to(device)
ft_model.load_state_dict(torch.load(os.path.join(checkpoints_dir, 'finetuned_model.pt'), map_location=device), strict=False)
ft_model.eval()
print("Models loaded successfully.")

# Load full 1152-d representations (720 samples) from 3.5.2 cache
if 'base_1152' not in globals() or len(base_1152) != 720:
    print("Loading 720-sample 1152-d representations from disk cache...")
    base_1152 = np.load(os.path.join(activations_dir, 'base_full_1152.npy'))
    ft_1152 = np.load(os.path.join(activations_dir, 'ft_full_1152.npy'))

print(f"Base 1152-d shape: {base_1152.shape}")
print(f"FT 1152-d shape:   {ft_1152.shape}")

def ablate_and_evaluate(model, acts_1152, labels, top_k_neurons, dataset_mean_acts):
    """
    Proxy inference ablation: splits the 1152-d vector, clamps top-k Text neurons, 
    passes through final classification heads, and returns per-class accuracy.
    """
    text_cls = acts_1152[:, 0:1024]
    faces = acts_1152[:, 1024:1088]
    specs = acts_1152[:, 1088:1152].copy()
    
    for n in top_k_neurons:
        specs[:, n] = dataset_mean_acts[n]
        
    with torch.no_grad():
        t_t = torch.tensor(text_cls, dtype=torch.float32).to(device)
        f_t = torch.tensor(faces, dtype=torch.float32).to(device)
        s_t = torch.tensor(specs, dtype=torch.float32).to(device)
        
        t_logits = model.t_out(t_t)
        v_logits = model.v_out(f_t)
        a_logits = model.a_out(s_t)
        
        all_logits = torch.stack([t_logits, v_logits, a_logits], dim=-1)
        final_logits = model.weighted_fusion(all_logits).squeeze(-1)
        preds = torch.argmax(final_logits, dim=1).cpu().numpy()
    
    accs = []
    for c in range(6):
        mask = (labels == c)
        acc = np.mean(preds[mask] == labels[mask]) if np.sum(mask) > 0 else 0.0
        accs.append(acc)
        
    return np.array(accs)

print("Ablation proxy inference scaffold ready.")

Loading PyTorch MME2E models for ablation...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json:   0%|          | 0.00/685 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/71.5M [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/torch/nn/modules/transformer.py:286: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  warnings.warn(f"enable_nested_tensor is True, but self.use_nested_tensor is False because {why_not_sparsity_fast_path}")


Models loaded successfully.
Base 1152-d shape: (720, 1152)
FT 1152-d shape:   (720, 1152)
Ablation proxy inference scaffold ready.


## 8. Execute Causal Selectivity Sweep & Evaluation

We now compute the baseline accuracy and evaluate the accuracy drop for ablating the top k in {1, 3, 5, 10} neurons.

According to ADR 0001, a feature set is causally class-selective if:
`Target Class Accuracy Drop` >= 2.5x `Mean Absolute Non-Target Class Drop`.

**Expected output:** Tables reporting the baseline accuracy, the ablation accuracy drops for k=5 (primary reporting target), and the final causal selectivity pass/fail per class, cleanly formatted. Artifacts (CSV and JSON) are saved to results.

In [30]:
import os
import json
import torch
import pandas as pd
import numpy as np
from src.ablation import generate_dose_response_ks, compute_per_class_accuracies

EMOTION_CLASSES = ['anger', 'disgust', 'fear', 'happiness', 'sadness', 'surprise']
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')

# Generate dose-response k values bounded by feature dimension (e.g. k=1, 3, 5, 10, 16, 32, 48, 64)
sweep_ks = generate_dose_response_ks(feature_dim=feature_dim)
print(f"Executing Dose-Response Sweep for k in: {sweep_ks}")

# Helper for proxy inference through PyTorch MME2E classification heads
def run_proxy_inference(model, acts_1152):
    text_cls = acts_1152[:, 0:1024]
    faces = acts_1152[:, 1024:1088]
    specs = acts_1152[:, 1088:1152]
    
    with torch.no_grad():
        t_t = torch.tensor(text_cls, dtype=torch.float32).to(device)
        f_t = torch.tensor(faces, dtype=torch.float32).to(device)
        s_t = torch.tensor(specs, dtype=torch.float32).to(device)
        
        t_logits = model.t_out(t_t)
        v_logits = model.v_out(f_t)
        a_logits = model.a_out(s_t)
        
        all_logits = torch.stack([t_logits, v_logits, a_logits], dim=-1)
        final_logits = model.weighted_fusion(all_logits).squeeze(-1)
        preds = torch.argmax(final_logits, dim=1).cpu().numpy()
    return preds

# Function to run ablation evaluation across full dataset (720 samples) with split reporting
def run_full_dose_response_sweep(model, full_1152, acts_tier1, labels, top_neurons_per_class, model_name):
    # Training mean for clamping (computed from train split 0:518)
    train_mean = np.mean(acts_tier1[0:518], axis=0)
    
    results_by_k = {}
    
    for k in sweep_ks:
        results_by_k[k] = {}
        for class_idx, class_name in enumerate(EMOTION_CLASSES):
            # Top-k neurons for this class (indexed by integer class_idx)
            target_neurons = top_neurons_per_class[class_idx][:k]
            
            # Copy feature matrix and apply mean clamp to targeted neurons
            patched_1152 = full_1152.copy()
            patched_tier1 = patched_1152[:, slice_start:slice_end]
            patched_tier1[:, target_neurons] = train_mean[target_neurons]
            
            # Proxy inference through model classification heads
            preds = run_proxy_inference(model, patched_1152)
            
            # Accuracies over full (0:720), train (0:518), and test (518:720) splits
            acc_full = compute_per_class_accuracies(preds, labels)
            acc_train = compute_per_class_accuracies(preds[0:518], labels[0:518])
            acc_test = compute_per_class_accuracies(preds[518:720], labels[518:720])
            
            results_by_k[k][class_name] = {
                'target_neurons': [int(n) for n in target_neurons],
                'full_dataset': acc_full,
                'train_split': acc_train,
                'test_split': acc_test,
            }
            
    return results_by_k

# Run sweeps for both models
base_sweep_results = run_full_dose_response_sweep(
    base_model, base_full_1152, base_acts_tier1, labels_tier1, base_top_neurons, 'base'
)

ft_sweep_results = run_full_dose_response_sweep(
    ft_model, ft_full_1152, ft_acts_tier1, labels_tier1, ft_top_neurons, 'finetuned'
)

# Export sweep JSON artifacts
os.makedirs('results', exist_ok=True)
with open(os.path.join('results', 'phase_c_base_ablation_sweep.json'), 'w') as f:
    json.dump(base_sweep_results, f, indent=2)

with open(os.path.join('results', 'phase_c_ft_ablation_sweep.json'), 'w') as f:
    json.dump(ft_sweep_results, f, indent=2)

print("Dose-response sweeps completed and exported to results/.")

Executing Dose-Response Sweep for k in: [1, 3, 5, 10, 16, 32, 48, 64]
Dose-response sweeps completed and exported to results/.


## 8.5 Day 13 Cosine Similarity Analysis (Issue A.3 & Protocol Day 13 Fix)

Compute the per-class selectivity vector `selectivity[d] = mean(act[d] | c) - mean(act[d] | ~c)` for both base and fine-tuned models, and calculate the representational cosine similarity across all 6 emotion classes.

**Expected output:** A 6-row summary table and saved CSV artifact `results/phase_d_cosine_similarity.csv`.

In [31]:
import os
import numpy as np
import pandas as pd
from scipy.spatial.distance import cosine
from utils import set_deterministic_seed

set_deterministic_seed(seed=0)
print('=== Section 8.5: Day 13 Cosine Similarity Analysis ===')

EMOTION_CLASSES = [
    'anger',
    'disgust',
    'fear',
    'happiness',
    'sadness',
    'surprise',
]


def compute_selectivity_vector(acts, labels, class_idx):
  target_mean = np.mean(acts[labels == class_idx], axis=0)
  off_target_mean = np.mean(acts[labels != class_idx], axis=0)
  return target_mean - off_target_mean


cosine_results = []
for c in range(6):
  class_name = EMOTION_CLASSES[c]

  base_sel = compute_selectivity_vector(base_acts_tier1, labels_tier1, c)
  ft_sel = compute_selectivity_vector(ft_acts_tier1, labels_tier1, c)

  # Cosine similarity = 1 - cosine distance
  sim = 1.0 - cosine(base_sel, ft_sel)

  cosine_results.append({
      'class': class_name,
      'cosine_similarity': round(sim, 4),
      'interpretation': (
          'High Sharpening' if sim >= 0.70 else 'Representation Shift/Rotate'
      ),
  })

df_cosine = pd.DataFrame(cosine_results)
print('\nBase vs Fine-Tuned Selectivity Vector Cosine Similarity:')
print(df_cosine.to_string(index=False))

out_cos_path = os.path.join(results_dir, 'phase_d_cosine_similarity.csv')
df_cosine.to_csv(out_cos_path, index=False)
print(f'\n[SAVED] {out_cos_path}')

=== Section 8.5: Day 13 Cosine Similarity Analysis ===

Base vs Fine-Tuned Selectivity Vector Cosine Similarity:
    class  cosine_similarity  interpretation
    anger             0.9048 High Sharpening
  disgust             0.7491 High Sharpening
     fear             0.9832 High Sharpening
happiness             0.7089 High Sharpening
  sadness             0.9570 High Sharpening
 surprise             0.9394 High Sharpening

[SAVED] /content/drive/MyDrive/multimodal-causal-ablation/results/phase_d_cosine_similarity.csv


# Phase D: Transfer Retention Analysis

Evaluate the base model's top-5 causally selective neurons inside the fine-tuned model to compute the Transfer Retention Ratio (R), following the methodology locked in [ADR 0001](docs/adr/0001-causal-validation-methodology.md).

**Goal:** Classify the fine-tuning outcome as Substrate Preservation (R >= 0.70), Substrate Reassignment (R < 0.30 with FT sparse drop), or Substrate Dispersion (R < 0.30 with FT dense drop).

## 9. Compute Transfer Retention Ratio (R)

Filter for emotion classes that proved causally selective in the base model (from Phase C) and evaluate those exact top-5 base neurons inside the fine-tuned model. The ratio of the fine-tuned accuracy drop to the base accuracy drop gives $R$, which determines the Substrate Outcome taxonomy.

In [36]:
# TRIAL
import os
import json
import torch
import pandas as pd
import numpy as np
from src.ablation import compute_transfer_retention_ratio, compute_per_class_accuracies

EMOTION_CLASSES = ['anger', 'disgust', 'fear', 'happiness', 'sadness', 'surprise']
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')

# 1. Unablated Baseline Accuracies (k=0) across full dataset (720 samples)
base_unablated_preds = run_proxy_inference(base_model, base_full_1152)
ft_unablated_preds   = run_proxy_inference(ft_model, ft_full_1152)

base_unablated_full = compute_per_class_accuracies(base_unablated_preds, labels_tier1)
ft_unablated_full   = compute_per_class_accuracies(ft_unablated_preds, labels_tier1)

# Training set means for clamping (computed from 0:518)
base_train_mean = np.mean(base_acts_tier1[0:518], axis=0)
ft_train_mean   = np.mean(ft_acts_tier1[0:518], axis=0)

# Evaluate Cross-Ablation over sweep ks: k in [1, 3, 5, 10, 16, 32, 48, 64]
sweep_ks = [1, 3, 5, 10, 16, 32, 48, 64]
dose_response_rows = []

for k in sweep_ks:
    for class_idx, class_name in enumerate(EMOTION_CLASSES):
        topk_base_neurons = base_top_neurons[class_idx][:k]
        
        # A) Base model self-ablation (top-k base neurons ablated in Base model)
        patched_base = base_full_1152.copy()
        patched_base[:, slice_start:slice_end][:, topk_base_neurons] = base_train_mean[topk_base_neurons]
        base_ablated_preds = run_proxy_inference(base_model, patched_base)
        base_ablated_full = compute_per_class_accuracies(base_ablated_preds, labels_tier1)
        base_drop = base_unablated_full[class_name] - base_ablated_full[class_name]
        
        # B) FT model cross-ablation (top-k base neurons ablated in FT model)
        patched_ft = ft_full_1152.copy()
        patched_ft[:, slice_start:slice_end][:, topk_base_neurons] = ft_train_mean[topk_base_neurons]
        ft_cross_preds = run_proxy_inference(ft_model, patched_ft)
        ft_cross_full = compute_per_class_accuracies(ft_cross_preds, labels_tier1)
        ft_cross_drop = ft_unablated_full[class_name] - ft_cross_full[class_name]
        
        # C) Retention ratio R = ft_cross_drop / base_drop
        ratio, taxonomy = compute_transfer_retention_ratio(base_drop, ft_cross_drop)
        
        dose_response_rows.append({
            'k': k,
            'emotion_class': class_name,
            'base_unablated_acc': round(base_unablated_full[class_name], 2),
            'base_drop': round(base_drop, 2),
            'ft_unablated_acc': round(ft_unablated_full[class_name], 2),
            'ft_cross_drop': round(ft_cross_drop, 2),
            'retention_ratio_R': round(ratio, 4) if ratio is not None else 'N/A',
            'taxonomy_outcome': taxonomy,
        })

df_dose = pd.DataFrame(dose_response_rows)
os.makedirs('results', exist_ok=True)
df_dose.to_csv(os.path.join('results', 'phase_d_dose_response_retention.csv'), index=False)

print("=== Full-Dataset Dose-Response Retention Analysis (N=720) ===")
# Display summary table for k=16 (dose-response knee point)
k16_df = df_dose[df_dose['k'] == 16][['emotion_class', 'base_drop', 'ft_cross_drop', 'retention_ratio_R', 'taxonomy_outcome']]
print("\nRetention at k=16 (Ablating 25% of Acoustic Neurons):")
print(k16_df.to_string(index=False))

# Display full modality knockout (k=64)
k64_df = df_dose[df_dose['k'] == 64][['emotion_class', 'base_drop', 'ft_cross_drop', 'retention_ratio_R', 'taxonomy_outcome']]
print("\nRetention at k=64 (Full Modality Knockout):")
print(k64_df.to_string(index=False))

=== Full-Dataset Dose-Response Retention Analysis (N=720) ===

Retention at k=16 (Ablating 25% of Acoustic Neurons):
emotion_class  base_drop  ft_cross_drop retention_ratio_R       taxonomy_outcome
        anger       0.83           5.00               6.0 Substrate Preservation
      disgust       0.00          -2.50               N/A     N/A (No Base Drop)
         fear      -5.00          -4.17            0.8333 Substrate Preservation
    happiness      -0.83          -2.50               3.0 Substrate Preservation
      sadness       0.83           0.00               0.0   Substrate Dispersion
     surprise       0.00          -1.67               N/A     N/A (No Base Drop)

Retention at k=64 (Full Modality Knockout):
emotion_class  base_drop  ft_cross_drop retention_ratio_R       taxonomy_outcome
        anger      18.33          11.67            0.6364 Substrate Reassignment
      disgust       4.17         -10.83              -2.6   Substrate Dispersion
         fear      -4.17    

In [34]:
import os
import json
import pandas as pd
from src.ablation import compute_transfer_retention_ratio

# Evaluate base model top-5 neurons inside fine-tuned model (Cross-Ablation)
# Using compute_transfer_retention_ratio to cleanly handle R = 0/0 edge cases

retention_rows = []

for class_name in EMOTION_CLASSES:
    # Get baseline accuracies (k=0) and k=5 accuracies from Phase C sweeps
    base_acc_k0 = base_sweep_results[1][class_name]['test_split']['overall'] # baseline
    base_acc_k5 = base_sweep_results[5][class_name]['test_split'][class_name]
    base_drop = base_acc_k0 - base_acc_k5
    
    ft_acc_k0 = ft_sweep_results[1][class_name]['test_split']['overall']
    ft_acc_k5 = ft_sweep_results[5][class_name]['test_split'][class_name]
    ft_drop = ft_acc_k0 - ft_acc_k5
    
    ratio, taxonomy = compute_transfer_retention_ratio(base_drop, ft_drop)
    
    retention_rows.append({
        'emotion_class': class_name,
        'base_drop': base_drop,
        'ft_drop': ft_drop,
        'retention_ratio_R': ratio if ratio is not None else 'N/A',
        'taxonomy_outcome': taxonomy,
    })

retention_df = pd.DataFrame(retention_rows)
retention_df.to_csv(os.path.join('results', 'phase_d_transfer_retention.csv'), index=False)

print("=== Phase D Transfer Retention Analysis ===")
print(retention_df.to_string(index=False))

=== Phase D Transfer Retention Analysis ===
emotion_class  base_drop    ft_drop  retention_ratio_R       taxonomy_outcome
        anger  -2.387886  16.627839          -6.963415   Substrate Dispersion
      disgust   0.553291  31.828771          57.526316 Substrate Preservation
         fear  -5.824112  10.745486          -1.845000   Substrate Dispersion
    happiness  35.847408  17.122889           0.477660 Substrate Reassignment
      sadness -10.726073 -42.334233           3.946853 Substrate Preservation
     surprise -16.291629 -38.313831           2.351750 Substrate Preservation
